# Project 2: Mouse gastrulation scRNA-seq (E6.5, E7.0, E7.5)
Pijuan-Sala et al. 2019 atlas (E-MTAB-6967), Scanpy + scVI.

**How to use:** Google Colab, run top to bottom. Use a **T4 GPU runtime** (Runtime > Change runtime type) before starting; only section 8 needs it.
All outputs go to `MyDrive/gastrulation_project/nb_run/` so they never overwrite earlier interactive files.

**Honest status:** assembled from the cells we ran interactively, with the known bugs fixed (no invalid 6,000-gene run, no `index` column, the Scrublet save fix). It has not yet been run end to end in one go. Hard-coded checks (`assert`) use numbers we verified earlier and will stop the run if something differs.

**Cluster numbers are not guaranteed to reproduce** across library versions. Section 9 applies cluster-to-label names from our run; always re-check them against the marker dot plots before trusting them.

**Rule kept throughout:** the authors' labels and QC flags are not opened until section 11.

## 0. Setup

In [ ]:
!pip install -q scanpy anndata igraph

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, sys, gzip, requests
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, scanpy as sc
import matplotlib.pyplot as plt
from importlib.metadata import version

proj = "/content/drive/MyDrive/gastrulation_project"
out = proj + "/nb_run"
base = "https://content.cruk.cam.ac.uk/jmlab/atlas_data/"
STAGES = ["E6.5", "E7.0", "E7.5"]
for d in ["metadata", "adata", "figures"]:
    os.makedirs(f"{out}/{d}", exist_ok=True)

# Shim: scanpy 1.12 calls csr_array.getnnz(axis=1), which scipy 1.16 no longer provides.
# np.diff(indptr) returns the same per-row stored-entry count. No data or parameters change.
def _getnnz(self, axis=None):
    if axis is None:
        return self.nnz
    if axis == 1:
        return np.diff(self.indptr)
    raise NotImplementedError("only axis=None or 1 supported in this shim")
sp.csr_array.getnnz = _getnnz

def save_fig(fig, name, download=False, dpi=300):
    """Save BEFORE plt.show(); set download=True to also download the PNG."""
    for ext in ["png", "pdf"]:
        fig.savefig(f"{out}/figures/{name}.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
    print("saved:", f"{out}/figures/{name}.png / .pdf")
    if download:
        files.download(f"{out}/figures/{name}.png")

def log(line):
    with open(out + "/analysis_log_nb.md", "a") as f:
        f.write("- " + line + "\n")

def ckpt(a, name):
    a.write_h5ad(f"{out}/adata/{name}.h5ad", compression="gzip")
    print("checkpoint:", name, a.shape)

def uniq(sym, genes):
    """Keep only markers whose symbol maps to exactly one Ensembl ID (78 genes share 37 symbols)."""
    bad = [g for g in genes if (sym == g).sum() != 1]
    print("missing/duplicated:", bad)
    return [g for g in genes if g not in bad]

for p in ["python", "pandas", "numpy", "scipy", "anndata", "scanpy"]:
    print(p, sys.version.split()[0] if p == "python" else version(p))
log("Environment: python %s, pandas %s, numpy %s, scipy %s, anndata %s, scanpy %s" % (
    sys.version.split()[0], version("pandas"), version("numpy"), version("scipy"), version("anndata"), version("scanpy")))

## 1. Dataset access and metadata
The authors' processed files are on their server (ArrayExpress E-MTAB-6967 holds FASTQs). The count matrix is 1.5 GB for the whole atlas, so we never load it whole.

In [ ]:
for f in ["raw_counts.mtx.gz", "genes.tsv.gz", "meta.tab.gz", "sizefactors.tab.gz"]:
    r = requests.head(base + f, allow_redirects=True, timeout=30)
    size = r.headers.get("Content-Length")
    print(f, r.status_code, "MB:", round(int(size) / 1e6, 1) if size else "unknown")

meta = pd.read_csv(base + "meta.tab.gz", sep="\t")
print(meta.shape)
print(meta.columns.tolist())

Split the metadata into three groups. The authors' annotations and QC flags are written to HIDDEN files and not opened again until section 11.

In [ ]:
design_cols  = ["cell", "barcode", "sample", "pool", "stage", "sequencing.batch", "theiler"]
authors_cols = ["cell", "cluster", "cluster.sub", "cluster.stage", "cluster.theiler", "celltype", "colour", "umapX", "umapY"]
qc_cols      = ["cell", "doub.density", "doublet", "stripped"]
for cols in (design_cols, authors_cols, qc_cols):
    assert all(c in meta.columns for c in cols), [c for c in cols if c not in meta.columns]

meta[design_cols].to_csv(f"{out}/metadata/design_metadata.csv", index=False)
meta[authors_cols].to_csv(f"{out}/metadata/authors_annotations_HIDDEN.csv", index=False)
meta[qc_cols].to_csv(f"{out}/metadata/authors_qc_HIDDEN.csv", index=False)

design = meta[design_cols]
print(design["stage"].value_counts())
print(design.groupby(["stage", "sample", "pool", "sequencing.batch"]).size())
print(pd.crosstab(design["stage"], design["theiler"]))   # theiler is a coarser copy of stage
log("Metadata split into design / authors annotations (HIDDEN) / authors QC (HIDDEN). File: 139,331 cells; sample 11 absent; mixed_gastrulation (TS9-10) is not one time point.")

## 2. Subset and stream the count matrix
Subset: E6.5, E7.0, E7.5 = 15 samples, 33,144 cells. `.mtx` is stored genes x cells; we read it in chunks, keep only our cells, and build cells x genes.

In [ ]:
design = pd.read_csv(f"{out}/metadata/design_metadata.csv")
assert design.shape[0] == 139331
assert (design["cell"].values == np.array([f"cell_{k+1}" for k in range(len(design))])).all(), "cell order mismatch"
keep_pos = np.where(design["stage"].isin(STAGES))[0]
sub = design.iloc[keep_pos]
print(sub.shape)
print(pd.crosstab(sub["stage"], sub["sequencing.batch"]))
assert len(keep_pos) == 33144

In [ ]:
npz = f"{out}/counts_E65_E70_E75_raw.npz"
if not os.path.exists(npz):
    !wget -q -O raw_counts.mtx.gz {base}raw_counts.mtx.gz
    with gzip.open("raw_counts.mtx.gz", "rt") as f:
        for _ in range(3):
            print(f.readline().strip())          # header: 29452 genes x 139331 cells, 483512215 non-zeros
    colmap = np.full(len(design) + 1, -1, dtype=np.int64)
    colmap[keep_pos + 1] = np.arange(len(keep_pos))
    R, C, V = [], [], []
    reader = pd.read_csv("raw_counts.mtx.gz", sep=" ", header=None, skiprows=2,
                         names=["g", "c", "v"], dtype=np.int32, chunksize=20_000_000)
    for i, ch in enumerate(reader):
        new_c = colmap[ch["c"].values]
        m = new_c >= 0
        R.append(ch["g"].values[m] - 1); C.append(new_c[m].astype(np.int32)); V.append(ch["v"].values[m])
        print("chunk", i, "kept", int(m.sum()))
    X = sp.csr_matrix((np.concatenate(V), (np.concatenate(C), np.concatenate(R))), shape=(len(keep_pos), 29452))
    sp.save_npz(npz, X)
X = sp.load_npz(npz)
print(X.shape, X.nnz, "| non-integer:", bool(np.any(X.data != np.round(X.data))), "| min/max:", X.data.min(), X.data.max())
assert X.shape == (33144, 29452) and X.nnz == 113965622
log("Counts streamed in 20M-line chunks, transposed to cells x genes; 113,965,622 non-zeros, integer, min 1, max 1284.")

## 3. Build the AnnData object (raw counts preserved)
Ensembl IDs are the index (unique); symbols stay as a column. `sample`, `pool`, `stage`, `sequencing.batch` are labels, so they become categories (a logged type change).

In [ ]:
genes = pd.read_csv(base + "genes.tsv.gz", sep="\t", header=None, names=["ensembl", "symbol"])
obs = design.iloc[keep_pos].reset_index(drop=True)
assert X.shape == (len(obs), len(genes))
assert genes["ensembl"].is_unique
print("symbols unique:", genes["symbol"].is_unique, "| genes sharing a symbol:", int(genes["symbol"].duplicated(keep=False).sum()))

adata = ad.AnnData(X=X, obs=obs.set_index("cell"), var=genes.set_index("ensembl"))
for col in ["sample", "pool", "stage", "sequencing.batch"]:
    adata.obs[col] = adata.obs[col].astype("category")
ckpt(adata, "01_raw_counts")
adata = ad.read_h5ad(f"{out}/adata/01_raw_counts.h5ad")        # always analyse the object reloaded from disk
print(adata.obs.groupby("stage", observed=True).size().to_dict(), "| total counts:", int(adata.X.sum()))
assert int(adata.X.sum()) == 551346183
log("01_raw_counts: 33,144 x 29,452, raw integer counts. Gene index = Ensembl IDs. 'barcode' is stored as category on save (unplanned type change).")

## 4. QC metrics and per-sample flags (nothing is removed)
The authors had already filtered the data (minimum 5,000 UMIs, maximum mito about 2.4%), so no low-quality tail is visible. Mito genes: the 13 protein-coding `mt-` genes. The 3-MAD rule assumes a symmetric distribution, which the 5,000-UMI floor breaks in low-depth samples, so flags are kept as columns only.

In [ ]:
adata.var["mt"] = adata.var["symbol"].str.startswith("mt-")
print(adata.var.loc[adata.var["mt"], "symbol"].tolist())
assert adata.var["mt"].sum() == 13
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

cols = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
print(adata.obs[cols].describe().round(2))
print(adata.obs.groupby("sample", observed=True)[cols].median().round(2))
sc.pl.violin(adata, keys=cols, groupby="sample", rotation=90, multi_panel=True, show=False)
save_fig(plt.gcf(), "qc_violin"); plt.show()

In [ ]:
from scipy.stats import median_abs_deviation as mad
x = np.log10(adata.obs["total_counts"]); g = adata.obs["sample"].astype(str)
med = x.groupby(g).transform("median")
dev = x.groupby(g).transform(lambda v: mad(v))
adata.obs["flag_hi"] = ((x - med) / dev > 3).values
adata.obs["flag_lo"] = ((x - med) / dev < -3).values
print(adata.obs.groupby(g)[["flag_lo", "flag_hi"]].sum())
print("flagged:", int(adata.obs["flag_lo"].sum() + adata.obs["flag_hi"].sum()), "of", adata.n_obs)
adata.obs["suspect_depth_samples"] = adata.obs["sample"].astype(str).isin(["14", "15", "18"]).values   # low-depth samples, for sensitivity runs only
ckpt(adata, "02_qc_metrics_no_filtering")
log("QC: counts min 5000, genes min 1412, mito max 2.37 (pre-filtered by authors). 3-MAD flags kept as columns, no cells removed. Samples 14/15/18 median 6.9k-9.2k UMI vs 13k-22k elsewhere (suspect_depth_samples).")

## 5. Doublet scoring (Scrublet, per sample, score only)
Expected rate 0.05 is the default, not tuned. Doublets are only simulated within a sample.

In [ ]:
sc.pp.scrublet(adata, batch_key="sample", expected_doublet_rate=0.05, random_state=0)
print(adata.obs.groupby("sample", observed=True)["predicted_doublet"].agg(["sum", "mean"]).round(3))
print("predicted doublets:", int(adata.obs["predicted_doublet"].sum()))

def strkeys(d):                       # h5ad needs string keys; numpy ints in uns['scrublet'] broke the write
    return {str(k): (strkeys(v) if isinstance(v, dict) else v) for k, v in d.items()}
if "scrublet" in adata.uns:
    adata.uns["scrublet"] = strkeys(adata.uns["scrublet"])
ckpt(adata, "03_scrublet_scores")
log("Scrublet per sample, rate 0.05, random_state 0: 189 predicted doublets (0.6%). Needed uns['scrublet'] keys converted to str for h5ad write.")

## 6. Normalisation, HVG-2000, PCA, neighbours, UMAP, Leiden
Library-size normalisation to 10,000 then log1p; raw counts stay in `layers['counts']`. HVGs use `batch_key='sequencing.batch'`. Scaling is done on a copy. The numbers 2,000 / 50 PCs / 30 PCs used / 15 neighbours / resolutions are conventions, not optimised.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
assert int(adata.layers["counts"].sum()) == 551346183

sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat", batch_key="sequencing.batch")
print("HVGs:", int(adata.var["highly_variable"].sum()))
print(adata.var.groupby("highly_variable_nbatches")["highly_variable"].sum())
print("mt genes among HVGs:", int((adata.var["highly_variable"] & adata.var["mt"]).sum()))

hv = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(hv, max_value=10)
sc.tl.pca(hv, n_comps=50, svd_solver="arpack", random_state=0)
adata.obsm["X_pca_hvg"] = hv.obsm["X_pca"]
adata.uns["pca_hvg_variance_ratio"] = hv.uns["pca"]["variance_ratio"]
sc.pl.pca_variance_ratio(hv, n_pcs=50, log=True, show=False)
save_fig(plt.gcf(), "pca_variance_ratio"); plt.show()
sc.pl.pca(hv, color=["stage", "sequencing.batch"], components=["1,2", "3,4"], ncols=2, show=False)
save_fig(plt.gcf(), "pca_hvg_stage_batch"); plt.show()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_hvg", key_added="nn_hvg", random_state=0)
sc.tl.umap(adata, neighbors_key="nn_hvg", random_state=0)
adata.obsm["X_umap_hvg"] = adata.obsm["X_umap"].copy()
for r in [0.3, 0.6, 1.0, 1.5]:
    sc.tl.leiden(adata, resolution=r, neighbors_key="nn_hvg", key_added=f"leiden_hvg_r{r}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
print(adata.obs[[c for c in adata.obs if c.startswith("leiden_hvg")]].nunique())
sc.pl.embedding(adata, basis="umap_hvg", color=["sequencing.batch", "stage", "flag_lo"], show=False)
save_fig(plt.gcf(), "umap_hvg_batch_stage_flag"); plt.show()
ckpt(adata, "04_hvg2000_leiden")
log("HVG-2000 (seurat, batch_key sequencing.batch; 741 genes in 3 batches, 710 in 2, 549 in 1; 7 mt genes among HVGs kept). PCA 50 comps (scaled copy, max_value 10), neighbors 15 / 30 PCs, UMAP and Leiden r0.3/0.6/1.0/1.5 (seed 0).")

**What to look at here.** Check which clusters are dominated by one sample (cluster x sample crosstab) and compare their median `total_counts`. In our run, one HVG cluster was ~98% samples 14/15/18 and had about half the depth of its neighbour. Cluster composition tracks sample and depth, but stage, pool and sample are confounded, so the cause could not be isolated.

In [ ]:
c = "leiden_hvg_r0.6"
samp = pd.crosstab(adata.obs[c], adata.obs["sample"], normalize="index")
tab = pd.DataFrame({"n": adata.obs[c].value_counts(), "top_sample": samp.idxmax(axis=1), "top_share": samp.max(axis=1).round(2),
                    "median_UMI": adata.obs.groupby(c, observed=True)["total_counts"].median().round(0),
                    "flag_lo": adata.obs.groupby(c, observed=True)["flag_lo"].mean().round(2)})
print(tab.sort_index(key=lambda s: s.astype(int)))

## 7. Broader gene set: HVG-6000 (corrected)
**Bug fixed here.** The object already carries `var['highly_variable']` for the 2,000-gene set. If it is copied into the subset, `sc.tl.pca` silently uses it and the "6,000-gene" PCA is really the 2,000-gene one. We set the flag True for all selected genes, pass `mask_var=None`, and verify.

In [ ]:
sel = sc.pp.highly_variable_genes(adata, n_top_genes=6000, flavor="seurat", batch_key="sequencing.batch", inplace=False)
b = adata[:, sel["highly_variable"].values].copy()
b.var["highly_variable"] = True
sc.pp.scale(b, max_value=10)
sc.tl.pca(b, n_comps=50, svd_solver="arpack", random_state=0, mask_var=None)
n_load = int((np.abs(b.varm["PCs"]).sum(axis=1) > 0).sum())
differs = not np.allclose(np.abs(b.obsm["X_pca"][:, :10]), np.abs(adata.obsm["X_pca_hvg"][:, :10]), atol=1e-3)
print("genes with PCA loadings:", n_load, "| differs from 2000-gene PCA:", differs)
assert n_load == 6000 and differs

adata.obsm["X_pca_hvg6000"] = b.obsm["X_pca"]
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_hvg6000", key_added="nn_hvg6000", random_state=0)
sc.tl.leiden(adata, resolution=0.6, neighbors_key="nn_hvg6000", key_added="leiden_hvg6000_r0.6",
             flavor="igraph", n_iterations=2, directed=False, random_state=0)
print("n clusters:", adata.obs["leiden_hvg6000_r0.6"].nunique())
ckpt(adata, "05_hvg6000")
log("HVG-6000 (corrected): flag set True for all 6000, mask_var=None, loadings checked. Earlier run that reused the 2000-gene flag was invalid and is not part of this notebook.")

## 8. scVI (needs the GPU runtime)
Input: raw counts of the 2,000 HVGs. `batch_key='sequencing.batch'` (3 batches). Using `sample` as batch would also remove stage signal, because each sample is one stage. Trained for 100 epochs, early stopping did not trigger, so convergence is not proven.

In [ ]:
!pip install -q scvi-tools

In [ ]:
import scvi, torch
print("scvi-tools", version("scvi-tools"), "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "switch to a GPU runtime"

b = adata[:, adata.var["highly_variable"]].copy()
assert b.n_vars == 2000
scvi.settings.seed = 0
scvi.model.SCVI.setup_anndata(b, layer="counts", batch_key="sequencing.batch")
model = scvi.model.SCVI(b, n_latent=30, n_layers=2, gene_likelihood="nb")
model.train(max_epochs=100, early_stopping=True)
adata.obsm["X_scVI"] = model.get_latent_representation()
model.save(f"{out}/scvi_model_hvg2000", overwrite=True)

h = model.history
print(h["elbo_train"].tail(3)); print(h["elbo_validation"].tail(3))
fig = plt.figure(figsize=(5, 3.5))
plt.plot(h["elbo_train"], label="train"); plt.plot(h["elbo_validation"], label="validation")
plt.ylim(850, 1100); plt.xlabel("epoch"); plt.ylabel("ELBO"); plt.legend()
save_fig(fig, "scvi_training_curve"); plt.show()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_scVI", key_added="nn_scvi", random_state=0)
sc.tl.umap(adata, neighbors_key="nn_scvi", random_state=0)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()
adata.obsm["X_umap"] = adata.obsm["X_umap_hvg"].copy()
sc.tl.leiden(adata, resolution=0.6, neighbors_key="nn_scvi", key_added="leiden_scvi_r0.6",
             flavor="igraph", n_iterations=2, directed=False, random_state=0)
print("n clusters:", adata.obs["leiden_scvi_r0.6"].nunique())
sc.pl.embedding(adata, basis="umap_scvi", color=["sequencing.batch", "stage"], show=False)
save_fig(plt.gcf(), "umap_scvi_batch_stage"); plt.show()
ckpt(adata, "06_scvi")
log("scVI 1.5.x: 2000 HVGs, layer counts, batch_key sequencing.batch, n_latent 30, n_layers 2, nb, 100 epochs (no early stop), seed 0. Latent in obsm['X_scVI'].")

## 9. Marker-based annotation (independent of the authors)
Dot plot colour is scaled per gene (`standard_scale='var'`), so dark means "highest among these clusters", not "high overall". Top-ranked Wilcoxon genes favour abundant genes, so they are a first look only.

In [ ]:
c = "leiden_scvi_r0.6"
sym = adata.var["symbol"]
n_expr = np.asarray((adata.layers["counts"] > 0).sum(axis=0)).ravel()
t = adata[:, n_expr >= 50].copy()
sc.tl.rank_genes_groups(t, c, method="wilcoxon", pts=True, key_added="de_scvi_r06")
rows = []
for g in t.obs[c].cat.categories:
    names = t.uns["de_scvi_r06"]["names"][g][:8]
    rows.append({"cluster": g, "n": int((t.obs[c] == g).sum()), "top_genes": ", ".join(adata.var.loc[names, "symbol"])})
print(pd.DataFrame(rows).to_string(index=False))

panel = {
 "epiblast": ["Pou5f1", "Dnmt3b", "Utf1", "Sox2", "Nanog"],
 "streak/mesoderm": ["T", "Mixl1", "Eomes", "Mesp1", "Fgf8", "Lhx1"],
 "endoderm": ["Foxa2", "Sox17", "Cer1", "Hhex"],
 "visceral endoderm": ["Ttr", "Apoa1", "Afp"],
 "extra-emb. ectoderm": ["Elf5", "Rhox5", "Bmp4"],
 "trophoblast": ["Plac1", "Prl3d1", "Plet1"],
 "ectoderm/neural": ["Otx2", "Six3", "Sox1", "Pax6"],
}
panel = {k: uniq(sym, v) for k, v in panel.items()}
sc.pl.dotplot(adata, panel, groupby=c, gene_symbols="symbol", standard_scale="var", show=False)
save_fig(plt.gcf(), "dotplot_markers_1"); plt.show()

In [ ]:
panel2 = {
 "ExE ectoderm": ["Elf5", "Eomes", "Cdx2", "Esrrb", "Tfap2c"],
 "posterior epiblast/streak": ["Nodal", "Wnt3", "Fgf8", "T"],
 "visceral endoderm": ["Hnf4a", "Gata4", "Gata6", "Dkk1", "Foxa1"],
 "mesoderm (lateral/ExM)": ["Hand1", "Bmp4", "Pdgfra", "Kdr"],
}
panel2 = {k: uniq(sym, v) for k, v in panel2.items()}
genes2 = [g for v in panel2.values() for g in v]
Xg = adata[:, [sym.index[sym == g][0] for g in genes2]].X
Xg = Xg.toarray() if sp.issparse(Xg) else np.asarray(Xg)
pct = pd.DataFrame(Xg > 0, columns=genes2, index=adata.obs[c].astype(str).values).groupby(level=0).mean().T * 100
print(pct.round(0).astype(int))        # % of cells expressing each gene, per cluster

**Provisional labels (annot_v2).** These cluster numbers come from our run. Re-check each against the dot plots and the percentage table before applying; if the numbering differs, rebuild the mapping from the markers (epiblast: Pou5f1/Dnmt3b/Utf1; streak: T/Mixl1/Eomes/Fgf8; ExE ectoderm: Elf5/Eomes/Cdx2/Tfap2c; visceral endoderm: Ttr/Apoa1/Afp). Labels are hypotheses; clusters 2, 9, 16 and 12 were uncertain or tiny.

In [ ]:
lab2 = {"0": "epiblast", "3": "epiblast", "10": "epiblast", "2": "posterior_epiblast_streak_cand",
        "1": "streak_mesoderm", "6": "mesoderm_cand", "15": "mesoderm_cand", "7": "mesoderm_cand",
        "4": "mesendoderm_cand", "16": "endoderm_cand", "8": "ExE_ectoderm", "11": "ExE_ectoderm", "14": "ExE_ectoderm",
        "5": "visceral_endoderm", "13": "visceral_endoderm", "9": "visceral_endoderm_cand", "12": "trophoblast_cand"}
adata.obs["annot_v2"] = adata.obs[c].astype(str).map(lab2).astype("category")
assert adata.obs["annot_v2"].isna().sum() == 0, "cluster numbering differs from our run: rebuild lab2 from the markers"
print(pd.crosstab(adata.obs["annot_v2"], adata.obs["stage"]))
sc.pl.embedding(adata, basis="umap_scvi", color="annot_v2", show=False)
save_fig(plt.gcf(), "umap_scvi_annot_v2"); plt.show()
ckpt(adata, "07_annot_v2")        # FROZEN before the authors' labels are opened
log("annot_v2 assigned from markers only (authors' files unopened); frozen in 07_annot_v2.")

## 10. Embryonic trajectory: PAGA and diffusion pseudotime
Only embryonic groups are used; extraembryonic cells follow different paths. PAGA edges show similarity of groups in the graph, not ancestry. The DPT **root is an assumption** (an E6.5 epiblast cell from samples 1 or 5 at the extreme of diffusion component 1, opposite the mesoderm candidates). It uses stage information, so agreement with stage is partly circular. The 153 endoderm candidates are dropped (no PAGA edges; they stretched the pseudotime scale and compressed all other groups into 0.10-0.118).

In [ ]:
from scipy.stats import spearmanr
def run_dpt(a, nn_key, root_name=None):
    sc.pp.neighbors(a, n_neighbors=15, use_rep="X_scVI", key_added=nn_key, random_state=0)
    sc.tl.diffmap(a, n_comps=15, neighbors_key=nn_key)
    if root_name is None:
        d = a.obsm["X_diffmap"][:, 1]
        ok = np.where((a.obs["stage"] == "E6.5") & (a.obs["annot_v2"] == "epiblast")
                      & a.obs["sample"].astype(str).isin(["1", "5"]))[0]
        meso = np.where(a.obs["annot_v2"] == "mesoderm_cand")[0]
        sign = np.sign(d[meso].mean() - d[ok].mean())
        root_name = a.obs_names[ok[np.argmin(sign * d[ok])]]
    a.uns["iroot"] = int(np.where(a.obs_names == root_name)[0][0])
    sc.tl.dpt(a, n_dcs=10, neighbors_key=nn_key)
    return root_name

def dpt_report(a, root_name):
    print("root:", root_name, "| sample", a.obs.loc[root_name, "sample"], "| cells:", a.n_obs)
    print(a.obs.groupby("stage", observed=True)["dpt_pseudotime"].quantile([.25, .5, .75]).unstack().round(3))
    print(a.obs.groupby("annot_v2", observed=True)["dpt_pseudotime"].median().round(3).sort_values())
    print("Spearman (rank-based):", round(spearmanr(a.obs["dpt_pseudotime"], a.obs["stage"].map({"E6.5": 0, "E7.0": 1, "E7.5": 2}).astype(int))[0], 3))

keep6 = ["epiblast", "posterior_epiblast_streak_cand", "streak_mesoderm", "mesoderm_cand", "mesendoderm_cand", "endoderm_cand"]
emb = adata[adata.obs["annot_v2"].isin(keep6)].copy()
emb.obs["annot_v2"] = emb.obs["annot_v2"].cat.remove_unused_categories()
print(emb.shape, emb.obs["annot_v2"].value_counts().to_dict())
sc.pp.neighbors(emb, n_neighbors=15, use_rep="X_scVI", key_added="nn_emb", random_state=0)
sc.tl.paga(emb, groups="annot_v2", neighbors_key="nn_emb")
sc.pl.paga(emb, threshold=0.05, show=False)     # threshold 0.05 is a display choice
save_fig(plt.gcf(), "paga_embryonic"); plt.show()

In [ ]:
e2 = emb[emb.obs["annot_v2"] != "endoderm_cand"].copy()
e2.obs["annot_v2"] = e2.obs["annot_v2"].cat.remove_unused_categories()
root_main = run_dpt(e2, "nn_emb2")
dpt_report(e2, root_main)
sc.pl.embedding(e2, basis="umap_scvi", color=["dpt_pseudotime", "stage"], show=False)
save_fig(plt.gcf(), "umap_scvi_dpt_main"); plt.show()
ckpt(e2, "08_dpt_main")
log("DPT main: embryonic groups without endoderm_cand, root E6.5 epiblast from samples 1/5 (stage-informed, circular vs real time).")

Sensitivity run: remove samples 14, 15, 18 and keep the same root cell. This is a labelled sensitivity analysis, not a replacement for the main run. In our run the cell-type order held; the fine E6.5 vs E7.0 separation depended on these samples.

In [ ]:
e3 = e2[~e2.obs["suspect_depth_samples"].values].copy()
assert root_main in e3.obs_names
run_dpt(e3, "nn_emb3", root_name=root_main)
dpt_report(e3, root_main)
ckpt(e3, "09_dpt_no_suspect_samples")
log("DPT sensitivity: samples 14/15/18 removed, same root cell.")

## 11. Compare with the authors' annotation (first time their files are opened)
`annot_v2` is frozen in `07_annot_v2`. The authors' labels are an independent annotation, not ground truth.

In [ ]:
a_ = pd.read_csv(f"{out}/metadata/authors_annotations_HIDDEN.csv")
q_ = pd.read_csv(f"{out}/metadata/authors_qc_HIDDEN.csv")
full = a_.merge(q_, on="cell")
print("file rows:", len(full), "| labelled:", int(full["celltype"].notna().sum()),
      "| stripped:", int(full["stripped"].sum()), "| doublet:", int(full["doublet"].sum()))
s_ = full.set_index("cell").loc[adata.obs_names]
adata.obs["auth_celltype"] = s_["celltype"].values
adata.obs["auth_doublet"] = s_["doublet"].astype(bool).values
adata.obs["auth_stripped"] = s_["stripped"].astype(bool).values
print("subset labelled:", int(adata.obs["auth_celltype"].notna().sum()), "| stripped:", int(adata.obs["auth_stripped"].sum()),
      "| doublet:", int(adata.obs["auth_doublet"].sum()))
print(pd.crosstab(adata.obs["predicted_doublet"], adata.obs["auth_doublet"]))

lab = adata.obs.dropna(subset=["auth_celltype"])
for k in sorted(lab["annot_v2"].unique()):
    top = lab.loc[lab["annot_v2"] == k, "auth_celltype"].value_counts(normalize=True).head(3).round(2)
    print(k, "| n =", int((lab["annot_v2"] == k).sum()), "|", top.to_dict())

adata.obs["auth_excluded"] = adata.obs["auth_stripped"] | adata.obs["auth_doublet"]
t = adata.obs.groupby("leiden_scvi_r0.6", observed=True).agg(n=("auth_excluded", "size"), stripped=("auth_stripped", "mean"),
        doublet=("auth_doublet", "mean"), flag_lo=("flag_lo", "mean")).round(2)
print(t.sort_values("stripped", ascending=False))
ckpt(adata, "10_with_authors_labels")
log("Authors' files opened after annot_v2 was frozen. Authors-excluded cells (stripped + doublet) are 11.8% of the subset and were present in all main analyses above.")

In [ ]:
from sklearn.metrics import adjusted_rand_score as ari, normalized_mutual_info_score as nmi
cl = ["leiden_hvg_r0.3", "leiden_hvg_r0.6", "leiden_hvg_r1.0", "leiden_hvg_r1.5", "leiden_hvg6000_r0.6", "leiden_scvi_r0.6"]
def score(d):
    return pd.DataFrame({"n_clusters": [d[c].nunique() for c in cl],
                         "ARI": [ari(d["auth_celltype"], d[c]) for c in cl],
                         "NMI": [nmi(d["auth_celltype"], d[c]) for c in cl]}, index=cl).round(2)
lab = adata.obs.dropna(subset=["auth_celltype"])
ns = lab[~lab["suspect_depth_samples"]]
print("all labelled cells:", len(lab)); print(score(lab))
print("without samples 14/15/18:", len(ns)); print(score(ns))

Rerun the trajectory without the authors-excluded cells (the authors' filtering used as an external QC reference).

In [ ]:
clean = adata[~adata.obs["auth_excluded"].values]
keep5 = ["epiblast", "posterior_epiblast_streak_cand", "streak_mesoderm", "mesoderm_cand", "mesendoderm_cand"]
e4 = clean[clean.obs["annot_v2"].isin(keep5)].copy()
e4.obs["annot_v2"] = e4.obs["annot_v2"].cat.remove_unused_categories()
root4 = run_dpt(e4, "nn_e4")
dpt_report(e4, root4)
ckpt(e4, "11_clean_embryonic_dpt")
log("Clean rerun (authors-excluded cells removed): cell-type order unchanged from the main run.")

## Limitations to keep in mind
Pool-level replication only (samples sharing a pool are probably repeat runs); stage, batch, pool and sample are partly confounded; the authors pre-filtered the data; our QC missed most authors-flagged stripped nuclei and doublets; definitive endoderm is under-sampled; one group (`posterior_epiblast_streak_cand`) was 40% rostral neurectoderm per the authors; all clustering parameters are conventions; ARI/NMI reward matching the authors' granularity; p-values from cell-level tests are inflated. See the decision-log document for the full list.